# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-5-pro'
MODEL_OLLAMA = 'gpt-oss:20b'

ollama = OpenAI(api_key="ollama", base_url='http://localhost:11434/v1')

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI()

API key looks good so far


In [ ]:
# here is the question; type over this to ask something new

question = """
I have this Jenkins script : 
"""

In [31]:
# Get gpt to answer, with streaming

def gpt_call():
    answer = openai.chat.completions.create(
        model = MODEL_GPT,
        messages = [
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in answer:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
gpt_call()

In [13]:
# Get Olama to answer

def ollama_call():
    answer = ollama.chat.completions.create(
        model = MODEL_OLLAMA,
        messages = [
            {"role": "user", "content": question}
          ],
    )    
    display(Markdown(answer.choices[0].message.content))

In [14]:
ollama_call()

### What the line actually does

```python
yield from {book.get("author") for book in books if book.get("author")}
```

1. **Build a *set comprehension***  
   ```python
   {book.get("author") for book in books if book.get("author")}
   ```
   * Iterates over every `book` in the iterable `books`.  
   * For each `book` it calls `book.get("author")`.  
   * The `if book.get("author")` clause keeps only those values that are *truthy* (i.e. not `None`, not an empty string, etc.).  
   * The result is a **set** of all *unique* author names that satisfy the condition.

2. **`yield from` de‑delegates to that set**  
   * `yield from` takes any (non‑generator) iterable and iterates over it, yielding each element to the caller of the surrounding generator.  
   * The set returned above is an iterable, so `yield from` will yield every author in the set, one by one.

So, in plain English:  
*“Take every book, grab its `author` field if it exists and is non‑empty, collect all the distinct authors, and then yield each of those authors to whoever is driving this generator.”*

---

### Why this construct is used

| Reason | How the code achieves it |
|--------|--------------------------|
| **Avoid an explicit `for` loop** | `yield from` is a concise way to forward all items from an iterable. |
| **Guarantee uniqueness** | Using a set comprehension eliminates duplicate authors automatically. |
| **Filter out missing/empty authors** | The `if book.get("author")` clause skips books that lack an `author` key or have a falsy value. |
| **Keep the generator tidy** | The line sits inside a larger generator function, letting the rest of the logic stay focused on higher‑level concerns. |

---

### Equivalent, more verbose form

If you prefer to see what `yield from` is doing behind the scenes, this line is absolutely identical to:

```python
unique_authors = {book.get("author") for book in books if book.get("author")}
for author in unique_authors:
    yield author
```

Both build the same set and then produce each author one after another.

---

### Caveats & Gotchas

1. **Order is unpredictable**  
   Sets in Python are unordered collections. The authors will be yielded in an arbitrary order that can change from run to run.  
   *If order matters*, replace the set with something ordered, e.g. `sorted(...)` or a `list` that you deduplicate manually.

2. **Only works inside a generator**  
   `yield` (and therefore `yield from`) can only appear inside a function defined with `def` (or a lambda in Python 3.9+). Running this line at module top‑level would raise a `SyntaxError`.

3. **Falsy author values are discarded**  
   `if book.get("author")` treats `''` (empty string), `0`, `False`, etc. as “missing”.  If you want to keep those, adjust the predicate (`if book.get("author") is not None`).

4. **No nested generators**  
   Here the object being delegated to is a plain set, not a generator.  That’s fine—`yield from` just iterates over any iterable—but its primary use case is delegating to sub‑generators.

---

### Bottom line

The code is a compact, generator‑friendly way to extract **unique, non‑empty author names** from a collection of books and stream them one by one. The combination of a set comprehension (for filtering & de‑duplication) with `yield from` (for delegating the yielding) keeps the generator body clean and expressive.